# Reconnaissance — what the Alteryx workflow has to reproduce

You cannot tell whether an Alteryx workflow is correct by looking at it. You can
only tell by comparing its output to a number you already trust.

This notebook runs all eight tests in pandas to establish those numbers. It is
also, precisely, the audit habit of independently re-performing a calculation
before accepting it.

**Run `01_generate_ledger.ipynb` first.**

| Container | What it proves |
|---|---|
| 1 — Completeness | The population is whole before anything is analysed |
| 2 — Standardise | Line items collapsed to entries, test fields derived |
| 3 — Eight tests | Each test's exception count |
| 4 — Scoring | Detection rate against the answer key |

## Load

In [1]:
import pandas as pd
import numpy as np

APPROVAL_LIMIT = 500_000
MATERIALITY    = 2_500_000

gl  = pd.read_csv("general_ledger.csv", parse_dates=["posting_date","effective_date"])
tb  = pd.read_csv("trial_balance.csv")
key = pd.read_csv("answer_key.csv")

print(f"ledger      {len(gl):>9,} lines")
print(f"trial bal   {len(tb):>9,} accounts")
print(f"answer key  {len(key):>9,} planted anomalies")

ledger        168,212 lines
trial bal          34 accounts
answer key        124 planted anomalies


---

# Container 1 — completeness

Prove the population is whole before analysing any of it. This is the step that
separates a defensible analysis from a plausible-looking one.

In [2]:
n_lines   = len(gl)
n_entries = gl["entry_id"].nunique()
dr, cr    = gl["debit"].sum(), gl["credit"].sum()

print(f"line items          {n_lines:>12,}")
print(f"journal entries     {n_entries:>12,}")
print(f"total debits        {dr:>16,.2f}")
print(f"total credits       {cr:>16,.2f}")
print(f"difference          {dr-cr:>16,.2f}   "
      f"{'BALANCED' if abs(dr-cr) < 0.01 else 'OUT OF BALANCE'}")
print()
print("nulls in key fields:")
for col in ["entry_id","posting_date","effective_date","account_code",
            "user_id","debit","credit"]:
    print(f"   {col:<18} {gl[col].isna().sum():>6,}")

line items               168,212
journal entries           84,106
total debits        30,523,085,595.19
total credits       30,523,085,595.19
difference                      0.00   BALANCED

nulls in key fields:
   entry_id                0
   posting_date            0
   effective_date          0
   account_code            0
   user_id                 0
   debit                   0
   credit                  0


## Period coverage — the finding that is sitting in the data

Check the two date fields separately. They do not agree, and the reason is the most
useful thing in this notebook.

In [3]:
FY_END = pd.Timestamp("2026-03-31")
late = gl[gl["posting_date"] > FY_END]

print(f"posting_date range     {gl['posting_date'].min().date()}  to  {gl['posting_date'].max().date()}")
print(f"effective_date range   {gl['effective_date'].min().date()}  to  {gl['effective_date'].max().date()}")
print()
print(f"lines posted after 31 March    {len(late):>5,}")
print(f"entries involved               {late['entry_id'].nunique():>5,}")
print()
late[["entry_id","posting_date","effective_date","narration","debit"]].head(6)

posting_date range     2025-04-01  to  2026-05-09
effective_date range   2025-04-01  to  2026-03-31

lines posted after 31 March       36
entries involved                  18



,entry_id,posting_date,effective_date,narration,debit
168176,JE1084046,2026-04-09,2026-03-31,Year end accrual,1284885.51
168177,JE1084046,2026-04-09,2026-03-31,Year end accrual,0.00
168178,JE1084060,2026-04-10,2026-03-30,Year end accrual,2538858.16
168179,JE1084060,2026-04-10,2026-03-30,Year end accrual,0.00
168180,JE1084047,2026-04-11,2026-03-30,Year end accrual,1116432.73
168181,JE1084047,2026-04-11,2026-03-30,Year end accrual,0.00


> **This is a real finding, not a data error.**
>
> Those 18 entries are year-end accruals: effective dates in March, posting dates in
> April and May. Entirely normal in practice.
>
> Which means **anyone who filters this population on posting date alone silently
> loses eighteen genuine year-end entries** — and gets a clean-looking analysis on an
> incomplete population. That is worse than no analysis, because it carries false
> confidence.

## Tie to the trial balance

Reconcile the dataset to its source of truth and explain any difference. An
unexplained difference is a finding, not a nuisance.

In [4]:
gl_by_acc = gl.groupby("account_code", as_index=False).agg(d=("debit","sum"), c=("credit","sum"))
chk = tb.merge(gl_by_acc, on="account_code", how="outer", indicator=True)
chk["diff"] = (chk["total_debit"] - chk["d"]).fillna(0) + (chk["total_credit"] - chk["c"]).fillna(0)

print(f"accounts in trial balance   {len(tb):>5,}")
print(f"accounts in ledger          {gl['account_code'].nunique():>5,}")
print(f"accounts not matching       {(chk['_merge'] != 'both').sum():>5,}")
print(f"maximum absolute difference {chk['diff'].abs().max():>9,.2f}")

accounts in trial balance      34
accounts in ledger             34
accounts not matching           0
maximum absolute difference      0.00


---

# Container 2 — standardise to entry level

The tests operate on entries, not lines. Collapse the two lines per entry into one
row and derive the fields each test needs.

In [5]:
ent = (gl.groupby("entry_id")
         .agg(posting_date=("posting_date","first"),
              effective_date=("effective_date","first"),
              user_id=("user_id","first"),
              entry_type=("entry_type","first"),
              narration=("narration","first"),
              approval_status=("approval_status","first"),
              amount=("debit","sum"),
              dr_acc=("account_code","first"),
              cr_acc=("account_code","last"))
         .reset_index())

HOLIDAYS = {"2025-04-14","2025-04-18","2025-05-01","2025-08-15","2025-08-27",
            "2025-10-02","2025-10-20","2025-10-21","2025-11-05","2025-12-25",
            "2026-01-26","2026-03-04"}

ent["day_num"]    = ent["posting_date"].dt.dayofweek            # 5,6 = weekend
ent["is_holiday"] = ent["posting_date"].dt.strftime("%Y-%m-%d").isin(HOLIDAYS)
ent["lag_days"]   = (ent["posting_date"] - ent["effective_date"]).dt.days
ent["acct_pair"]  = ent["dr_acc"].astype(str) + " / " + ent["cr_acc"].astype(str)
ent["seq_num"]    = ent["entry_id"].str.replace("JE","", regex=False).astype(int)

print(f"entries      {len(ent):,}")
print(f"manual       {(ent.entry_type=='Manual').sum():,}")
print(f"system       {(ent.entry_type=='System').sum():,}")
ent.head(3)

entries      84,106
manual       15,257
system       68,849


,entry_id,posting_date,effective_date,user_id,entry_type,narration,approval_status,amount,dr_acc,cr_acc,day_num,is_holiday,lag_days,acct_pair,seq_num
0,JE1000001,2026-03-19,2026-03-18,SYS-BATCH,System,Supplier payment,Not required,381444.79,2100,1110,3,False,1,2100 / 1110,1000001
1,JE1000002,2025-05-06,2025-05-06,U009,Manual,Sales invoice raised,Approved,536805.15,1200,4100,1,False,0,1200 / 4100,1000002
2,JE1000003,2026-03-03,2026-03-02,SYS-BATCH,System,Travel reimbursement,Not required,48308.91,5330,1110,1,False,1,5330 / 1110,1000003


---

# Container 3 — the eight tests

Each cell states the Alteryx equivalent, so you can build the workflow to match.

### T1 — Round numbers above materiality

`Filter: [amount] >= 2500000 AND MOD([amount],100000) = 0 AND [entry_type] = "Manual"`

In [6]:
results = {}

t1 = ent[(ent.amount >= MATERIALITY) &
         (ent.amount % 100_000 == 0) &
         (ent.entry_type == "Manual")]
results["round_number_above_materiality"] = t1

print(f"exceptions: {len(t1)}")
t1[["entry_id","posting_date","user_id","amount","narration"]].head()

exceptions: 14


,entry_id,posting_date,user_id,amount,narration
83991,JE1084001,2025-06-27,U006,10000000.0,Balance transfer
83992,JE1084002,2025-09-09,U003,10000000.0,Balance transfer
83993,JE1084003,2026-01-05,U009,2500000.0,Balance transfer
83994,JE1084004,2025-12-30,U001,10000000.0,Balance transfer
83995,JE1084005,2026-03-16,U001,5000000.0,Balance transfer


### T2 — Non-business-day postings

`Filter: ([day_num]=0 OR [day_num]=6 OR [is_holiday]=1) AND [entry_type]="Manual"`

Note pandas uses Monday=0, Alteryx's `%w` uses Sunday=0. Adjust accordingly.

In [7]:
t2 = ent[((ent.day_num >= 5) | ent.is_holiday) & (ent.entry_type == "Manual")]
results["non_business_day_posting"] = t2

print(f"exceptions: {len(t2)}")
print(f"planted:    {(key.anomaly_type=='non_business_day_posting').sum()}")
print()
print("The extra flags are entries planted for other reasons that happen to fall")
print("on a weekend. One entry can trip several tests -- which is why the scoring")
print("container ranks by how many tests fired.")

exceptions: 29
planted:    22

The extra flags are entries planted for other reasons that happen to fall
on a weekend. One entry can trip several tests -- which is why the scoring
container ranks by how many tests fired.


### T3 — Rare user IDs

Three tools in Alteryx: **Summarize** (Group By user_id, Count) → **Filter**
(Count < 10) → **Join** back on user_id. Learn this pattern; it recurs constantly.

In [8]:
counts     = ent.groupby("user_id").size().rename("n_entries")
rare_users = counts[counts < 10].index.tolist()
t3 = ent[ent.user_id.isin(rare_users)]
results["rare_user"] = t3

print(f"rare users: {rare_users}")
print(f"exceptions: {len(t3)}")
print()
print("entries per user (lowest 6):")
print(counts.sort_values().head(6).to_string())

rare users: ['U087', 'U091', 'U104']
exceptions: 9

entries per user (lowest 6):
user_id
U091      2
U087      3
U104      4
U015    662
U021    670
U016    687


### T4 — Back-dated entries

`Filter: DateTimeDiff([posting_date],[effective_date],"days") > 7`

In [9]:
t4 = ent[ent.lag_days > 7]
results["back_dated"] = t4

print(f"exceptions: {len(t4)}")
print(f"max lag:    {ent.lag_days.max()} days")
print()
print("These are the same entries flagged by the Container 1 completeness check --")
print("a useful cross-check that both are working.")

exceptions: 18
max lag:    41 days

These are the same entries flagged by the Container 1 completeness check --
a useful cross-check that both are working.


### T5 — Amounts below the approval threshold

**Build the naive version first.** It is instructive.

In [10]:
naive = ent[(ent.amount >= 0.94*APPROVAL_LIMIT) & (ent.amount < APPROVAL_LIMIT)]
planted_t5 = set(key[key.anomaly_type=="below_approval_threshold"].entry_id)

print(f"naive filter (94-100% of limit, all entries)")
print(f"   exceptions: {len(naive):,}")
print(f"   caught:     {len(set(naive.entry_id) & planted_t5)}")
print(f"   precision:  {len(set(naive.entry_id) & planted_t5)/len(naive):.1%}")
print()
print("by entry type:")
print(naive.entry_type.value_counts().to_string())

naive filter (94-100% of limit, all entries)
   exceptions: 2,231
   caught:     26
   precision:  1.2%

by entry type:
entry_type
System    1786
Manual     445


2,231 exceptions to find 26 real ones. Nobody will work through that list, and a
test nobody works through is a test that quietly dies.

The problem is not the band. It is the entry type.

In [11]:
designs = {
    "94-100%, all entries":
        ent[(ent.amount>=0.94*APPROVAL_LIMIT)&(ent.amount<APPROVAL_LIMIT)],
    "94-100%, manual only":
        ent[(ent.amount>=0.94*APPROVAL_LIMIT)&(ent.amount<APPROVAL_LIMIT)&(ent.entry_type=="Manual")],
    "98-100%, manual only":
        ent[(ent.amount>=0.98*APPROVAL_LIMIT)&(ent.amount<APPROVAL_LIMIT)&(ent.entry_type=="Manual")],
    "90-100%, manual only":
        ent[(ent.amount>=0.90*APPROVAL_LIMIT)&(ent.amount<APPROVAL_LIMIT)&(ent.entry_type=="Manual")],
}

print(f"{'design':<26}{'flagged':>10}{'caught':>9}{'missed':>9}{'precision':>12}")
print("-"*66)
for name, df in designs.items():
    caught = set(df.entry_id) & planted_t5
    print(f"{name:<26}{len(df):>10,}{len(caught):>9}{len(planted_t5)-len(caught):>9}"
          f"{len(caught)/len(df):>11.1%}")

design                       flagged   caught   missed   precision
------------------------------------------------------------------
94-100%, all entries           2,231       26        0       1.2%
94-100%, manual only             445       26        0       5.8%
98-100%, manual only             130        7       19       5.4%
90-100%, manual only             720       26        0       3.6%


> **The threshold design finding**
>
> Restricting to manual entries drops the exception list from 2,231 to 445 and still
> catches **all 26** planted items — five times the precision at zero cost in detection.
>
> Tightening the band to 98% instead cuts the list to 130 but loses 19 of the 26. A
> much worse trade.
>
> So the band was never the problem; the entry type was. Two hundred exceptions is a
> failed design. Forty exceptions with a documented reason for the cut-off is a working
> control — and the rationale is what makes it defensible, not the number.

In [12]:
t5 = designs["94-100%, manual only"]
results["below_approval_threshold"] = t5
print(f"adopted: 94-100% of limit, manual entries only -> {len(t5)} exceptions")

adopted: 94-100% of limit, manual entries only -> 445 exceptions


### T6 — Rare account pairings

Same three-tool pattern as T3, on `acct_pair`.

In [13]:
pair_counts = ent.groupby("acct_pair").size().rename("n")
rare_pairs  = pair_counts[pair_counts < 5].index
t6 = ent[ent.acct_pair.isin(rare_pairs)]
results["rare_account_pairing"] = t6

print(f"distinct rare pairings: {len(rare_pairs)}")
print(f"exceptions:             {len(t6)}")
print()
print(pair_counts[pair_counts < 5].to_string())

distinct rare pairings: 4
exceptions:             8

acct_pair
1510 / 5700    2
2400 / 4110    2
3200 / 1110    2
4100 / 1900    2


### T7 — Narration keywords

`Filter: REGEX_Match(Lowercase([narration]), ".*(plug|difference|reversal|...).*")`

In [14]:
KEYWORDS = ["plug","difference","reversal","temporary","pending","adjustment",
            "squaring","correction","as per instruction","to be corrected"]
t7 = ent[ent.narration.str.lower().str.contains("|".join(KEYWORDS), na=False)]
results["narration_keyword"] = t7

print(f"exceptions: {len(t7)}")
print(f"planted:    {(key.anomaly_type=='narration_keyword').sum()}")
print()
print("distinct narrations flagged:")
print(t7.narration.value_counts().to_string())

exceptions: 26
planted:    18

distinct narrations flagged:
narration
Adjustment as discussed                        8
To plug difference in control account          3
Reversal of earlier entry - to be corrected    3
Temporary posting pending confirmation         3
Adjustment to match management figures         3
Squaring off old balance                       3
Correction entry as per instruction            3


### T8 — Sequence gaps

**This is the one to make visible in your screenshots.** In Alteryx it needs the
**Multi-Row Formula** tool, which compares a row to the one before it:

`[seq_num] - [Row-1:seq_num] - 1`

Most people do not realise a visual tool can do that, which is why it signals real
Alteryx competence.

In [15]:
ids  = sorted(ent.seq_num)
full = set(range(min(ids), max(ids)+1))
gaps = sorted(full - set(ids))
results["sequence_gap"] = pd.DataFrame({"entry_id":[f"JE{g}" for g in gaps]})

print(f"gaps found: {len(gaps)}")
print(f"missing:    {[f'JE{g}' for g in gaps]}")

gaps found: 9
missing:    ['JE1020256', 'JE1025976', 'JE1048843', 'JE1050411', 'JE1051574', 'JE1061168', 'JE1063270', 'JE1063366', 'JE1074758']


---

# Container 4 — scoring against the answer key

This is what makes the project different from every other portfolio piece: it can
state what it **missed**, not just what it found.

In [16]:
flagged = set()
for df in results.values():
    flagged |= set(df["entry_id"])

planted = set(key.entry_id)
caught  = planted & flagged
missed  = planted - flagged

print(f"unique entries flagged   {len(flagged):>6,}")
print(f"planted anomalies        {len(planted):>6,}")
print(f"caught                   {len(caught):>6,}   ({len(caught)/len(planted):.1%})")
print(f"missed                   {len(missed):>6,}")

unique entries flagged      542
planted anomalies           124
caught                      124   (100.0%)
missed                        0


In [17]:
rows = []
for name, df in results.items():
    p = set(key[key.anomaly_type == name].entry_id)
    c = p & set(df["entry_id"])
    rows.append({
        "test": name,
        "flagged": len(df),
        "planted": len(p),
        "caught": len(c),
        "detection": f"{len(c)/len(p):.0%}" if len(p) else "-",
        "false_positives": len(df) - len(c),
    })

pd.DataFrame(rows).sort_values("flagged", ascending=False).reset_index(drop=True)

,test,flagged,planted,caught,detection,false_positives
0,below_approval_threshold,445,26,26,100%,419
1,non_business_day_posting,29,22,22,100%,7
2,narration_keyword,26,18,18,100%,8
3,back_dated,18,18,18,100%,0
4,round_number_above_materiality,14,14,14,100%,0
5,rare_user,9,9,9,100%,0
6,sequence_gap,9,9,9,100%,0
7,rare_account_pairing,8,8,8,100%,0


## Risk ranking

Entries tripping several tests deserve attention first. Write down why the cut-offs
are 3 and 2 — a threshold without a documented rationale is an opinion.

In [18]:
allf = pd.concat([df.assign(test_name=name)[["entry_id","test_name"]]
                  for name, df in results.items()])

scored = (allf.groupby("entry_id")
            .agg(tests_triggered=("test_name","count"),
                 tests=("test_name", lambda s: ", ".join(sorted(set(s)))))
            .reset_index())

scored["risk_rank"] = np.where(scored.tests_triggered >= 3, "High",
                       np.where(scored.tests_triggered == 2, "Medium", "Low"))

print("exceptions by risk rank:")
print(scored.risk_rank.value_counts().to_string())
print()
print("entries tripping more than one test:")
scored[scored.tests_triggered > 1].sort_values("tests_triggered", ascending=False).head(10)

exceptions by risk rank:
risk_rank
Low       526
Medium     16

entries tripping more than one test:


,entry_id,tests_triggered,tests,risk_rank
473,JE1084047,2,"back_dated, non_business_day_posting",Medium
479,JE1084053,2,"back_dated, non_business_day_posting",Medium
480,JE1084054,2,"back_dated, non_business_day_posting",Medium
483,JE1084057,2,"back_dated, non_business_day_posting",Medium
484,JE1084058,2,"back_dated, non_business_day_posting",Medium
487,JE1084061,2,"back_dated, non_business_day_posting",Medium
488,JE1084062,2,"back_dated, non_business_day_posting",Medium
516,JE1084090,2,"narration_keyword, rare_account_pairing",Medium
517,JE1084091,2,"narration_keyword, rare_account_pairing",Medium
518,JE1084092,2,"narration_keyword, rare_account_pairing",Medium


---

## Targets for the Alteryx build

| Container | Check | Target |
|---|---|---|
| 1 | Line items | 168,212 |
| 1 | Journal entries | 84,106 |
| 1 | Debits = credits | 30,523,085,595.19 |
| 1 | Lines posted after 31 Mar | 36 (18 entries) |
| 2 | Rows after collapse | 84,106 |
| 3 | T1 round numbers | 14 |
| 3 | T2 non-business day | 29 |
| 3 | T3 rare users | 9 |
| 3 | T4 back-dated | 18 |
| 3 | T5 below threshold (manual only) | 445 |
| 3 | T6 rare pairings | 8 |
| 3 | T7 narration keywords | 26 |
| 3 | T8 sequence gaps | 9 |
| 4 | Planted anomalies caught | 124 of 124 |

**If your workflow catches fewer than 124, do not tune the tests until it does.**
Find out which test missed which anomaly and why. A workflow catching 118 with a
clear explanation of the six misses is a better artifact than one tuned by trial and
error — and an interviewer can tell the difference in about thirty seconds.

---

*Synthetic data generated for this project. No client data is used.*